# AM01 Colab Master Notebook

Notebook operativo per eseguire il progetto AM01 su Google Colab. Il notebook non reimplementa la pipeline: richiama gli script del repository, salva risultati su Google Drive e produce gli artefatti necessari per analisi, report e presentazione.

Flusso consigliato:
1. setup ambiente;
2. test del repository;
3. audit dataset reale;
4. preprocessing leakage-safe;
5. training baseline e AE/AAE;
6. figure e tabelle;
7. ablation opzionale.

## 0. Parametri

Prima di eseguire tutto, imposta `REPO_URL` se vuoi clonare il repository da GitHub. Il dataset reale deve essere disponibile in Drive con la struttura `KukaColumnNames.npy`, `KukaNormal.npy`, `KukaSlow.npy`.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Bernuz2003/AML_anomaliy_detection.git"  # esempio: "https://github.com/<user>/am01-kuka-aae-anomaly-detection.git"
PROJECT_DIR = Path("/content/am01-kuka-aae-anomaly-detection")
DRIVE_ROOT = Path("/content/drive/MyDrive/AM01")
DATA_DIR = DRIVE_ROOT / "data" / "KukaVelocityDataset"
RESULTS_DIR = DRIVE_ROOT / "results"
PROCESSED_DIR = DRIVE_ROOT / "data" / "processed" / "kuka_default"

RUN_DEEP_MODELS = True
RUN_CONV1D = True
RUN_ABLATION = False  # attiva solo dopo aver validato il run principale

print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## 1. Mount Drive, clone/install e verifica ambiente

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
import subprocess

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise RuntimeError("PROJECT_DIR non esiste. Imposta REPO_URL o carica/clona il repo in /content.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

def sh(cmd: str) -> None:
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
!python --version
!pip install -q -r requirements.txt
!python -m compileall -q src scripts tests

## 2. Test del repository

Questa sezione controlla che il codice funzioni prima di usare il dataset reale.

In [ ]:
!pytest -q

## 3. Verifica dataset reale

In [ ]:
required = ["KukaColumnNames.npy", "KukaNormal.npy", "KukaSlow.npy"]
missing = [name for name in required if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"File mancanti in {DATA_DIR}: {missing}")
print("Dataset trovato:")
for name in required:
    path = DATA_DIR / name
    print(name, path.stat().st_size, "bytes")

## 4. Data audit ed esempi di segnali

In [ ]:
!python scripts/audit_data.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{RESULTS_DIR / 'data_audit'}"
!python scripts/plot_data_examples.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{RESULTS_DIR / 'figures' / 'data_examples'}"

In [ ]:
import pandas as pd

summary_path = RESULTS_DIR / "data_audit" / "dataset_summary.csv"
summary = pd.read_csv(summary_path)
display(summary[["run_id", "n_rows", "run_label", "anomaly_fraction", "source_file"]].head())
print("Runs:", len(summary))
print(summary["run_label"].value_counts(dropna=False))

## 5. Preprocessing leakage-safe

Crea finestre train/validation/test, scaler e summary. Lo scaler viene fitatto solo sui normali di training.

In [ ]:
!python scripts/prepare_data.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{PROCESSED_DIR}"
!ls -lh "{PROCESSED_DIR}"

## 6. Run principale: PCA, Isolation Forest, AE, AAE e Conv1D-AE

In [ ]:
MAIN_RUNS_DIR = RESULTS_DIR / "runs" / "main"
MAIN_RUNS_DIR.mkdir(parents=True, exist_ok=True)

pca_dir = MAIN_RUNS_DIR / "pca"
iforest_dir = MAIN_RUNS_DIR / "isolation_forest"
ae_dir = MAIN_RUNS_DIR / "ae_mlp"
aae_dir = MAIN_RUNS_DIR / "aae_mlp"
conv_dir = MAIN_RUNS_DIR / "ae_conv1d"

sh(f'python scripts/train.py --config configs/pca.yaml --data "{DATA_DIR}" --output "{pca_dir}"')
sh(f'python scripts/train.py --config configs/isolation_forest.yaml --data "{DATA_DIR}" --output "{iforest_dir}"')

if RUN_DEEP_MODELS:
    sh(f'python scripts/train.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{ae_dir}"')
    sh(f'python scripts/train.py --config configs/aae_mlp.yaml --data "{DATA_DIR}" --output "{aae_dir}"')
if RUN_CONV1D:
    sh(f'python scripts/train.py --config configs/ae_conv1d.yaml --data "{DATA_DIR}" --output "{conv_dir}"')

## 7. Tabelle metriche e figure

In [ ]:
import json

rows = []
for metrics_path in sorted(MAIN_RUNS_DIR.glob("*/metrics.json")):
    with metrics_path.open() as f:
        metrics = json.load(f)
    row = {"run": metrics_path.parent.name, "threshold": metrics.get("threshold")}
    for key, value in metrics.get("test_metrics", {}).items():
        row[f"test_{key}"] = value
    rows.append(row)

metrics_table = pd.DataFrame(rows).sort_values("run")
tables_dir = RESULTS_DIR / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
metrics_csv = tables_dir / "main_metrics.csv"
metrics_table.to_csv(metrics_csv, index=False)
display(metrics_table)
print("Saved:", metrics_csv)

In [ ]:
for run_dir in sorted(MAIN_RUNS_DIR.iterdir()):
    if run_dir.is_dir() and (run_dir / "scores_test.csv").exists():
        sh(f'python scripts/plot_results.py --run-dir "{run_dir}"')

for name in ["ae_mlp", "aae_mlp", "ae_conv1d"]:
    run_dir = MAIN_RUNS_DIR / name
    if run_dir.exists() and (run_dir / "model.pt").exists():
        sh(f'python scripts/plot_model_diagnostics.py --run-dir "{run_dir}" --split test')

## 8. Ablation opzionale

Da eseguire solo dopo aver verificato il run principale. Produce una tabella `experiment_summary.csv`.

In [ ]:
if RUN_ABLATION:
    ABLATION_DIR = RESULTS_DIR / "runs" / "ablation"
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/ae_mlp.yaml configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{ABLATION_DIR}" '
        f'--seeds 0 1 2 '
        f'--window-lengths 32 64 128 '
        f'--latent-dims 8 16 32 '
        f'--lambda-advs 0.01 0.1 1.0'
    )
else:
    print("Ablation disattivata. Imposta RUN_ABLATION=True nella sezione parametri.")

## 9. Manifest risultati

Usa questa lista per scaricare o copiare gli artefatti da Drive nel report.

In [ ]:
!find "{RESULTS_DIR}" -maxdepth 5 -type f | sort | sed -n '1,240p'